In [2]:
import torch
print("GPU Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("Device Name:", torch.cuda.get_device_name(0))

GPU Available: True
Device Name: Tesla T4


In [4]:
import torch
import torch.nn as nn
import math

class CustomLoRALinear(nn.Module):
    def __init__(self, in_features, out_features, r=8, lora_alpha=16):
        super(CustomLoRALinear, self).__init__()

        # 1. The original base linear layer (representing our frozen pre-trained model)
        self.linear = nn.Linear(in_features, out_features)

        # 2. Hyperparameters for rank and scaling factor
        self.r = r
        self.lora_alpha = lora_alpha
        self.scaling = lora_alpha / r

        # 3. Only instantiate LoRA matrices if rank > 0
        if r > 0:
            # Matrix A down-projects from in_features to r
            self.lora_A = nn.Parameter(torch.zeros((r, in_features)))
            # Matrix B up-projects from r to out_features
            self.lora_B = nn.Parameter(torch.zeros((out_features, r)))

            # 4. Critical initialization strategy from the paper:
            # Kaiming uniform for A, and zeros for B so that Delta W starts at 0!
            nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
            nn.init.zeros_(self.lora_B)

        # 5. Freeze the core base weights
        self.linear.weight.requires_grad = False

    def forward(self, x):
        # Base layer forward pass: W0 * x
        base_output = self.linear(x)

        if self.r > 0:
            # LoRA forward pass equation: h = W0*x + ((alpha/r) * B * A * x)
            # We use matrix multiplication (@) and transpose weights to align dimensions
            lora_output = (x @ self.lora_A.t() @ self.lora_B.t()) * self.scaling
            return base_output + lora_output

        return base_output

In [5]:
# Create dummy input: Batch size = 2, Sequence length = 5, Input dimensions = 128
dummy_input = torch.randn(2, 5, 128).cuda()

# Initialize our custom LoRA layer mapping 128 dims to 64 dims with rank r=4
lora_layer = CustomLoRALinear(in_features=128, out_features=64, r=4).cuda()

# Pass data through the layer
output = lora_layer(dummy_input)

print("Input shape:", dummy_input.shape)
print("Output shape:", output.shape)
print("Are base linear weights frozen?", not lora_layer.linear.weight.requires_grad)
print("Is lora_A trainable?", lora_layer.lora_A.requires_grad)

Input shape: torch.Size([2, 5, 128])
Output shape: torch.Size([2, 5, 64])
Are base linear weights frozen? True
Is lora_A trainable? True


In [11]:
from transformers import AutoModelForCausalLM, AutoTokenizer

# 1. Load a lightweight pre-trained model (GPT-2 is perfect for a free T4 GPU tier)
model_name = "gpt2"
print(f"Loading {model_name}...")
base_model = AutoModelForCausalLM.from_pretrained(model_name).cuda()

# 2. Function to recursively replace standard Conv1D/Linear layers with our Custom LoRA
def apply_lora_surgery(model, target_modules=["c_attn"], r=4, lora_alpha=8):
    """
    Scans the model architecture and replaces specific target layers with CustomLoRALinear.
    In GPT-2, the attention layers are stored as 'c_attn' (Hugging Face Conv1D layers).
    """
    for name, module in model.named_children():
        # If the child module contains sub-modules, recurse down the tree
        if len(list(module.children())) > 0:
            apply_lora_surgery(module, target_modules, r, lora_alpha)

        # Check if this module matches one of our target injection sites
        if name in target_modules:
            # For GPT-2's c_attn, out_features is the combined Q, K, V dimensions (3 * embed_dim)
            in_features = module.weight.shape[0]
            out_features = module.weight.shape[1]

            # Create our custom LoRA layer
            lora_layer = CustomLoRALinear(in_features, out_features, r=r, lora_alpha=lora_alpha).cuda()

            # (Optional) For strict math mapping, copy the original pre-trained weights over
            # Since GPT-2 uses transposed Conv1D weights, we align them to standard linear weights
            with torch.no_grad():
                lora_layer.linear.weight.copy_(module.weight.t())
                if module.bias is not None:
                    lora_layer.linear.bias.copy_(module.bias)

            # Complete the surgery by overwriting the old module on the parent class
            setattr(model, name, lora_layer)

# Run the surgery
apply_lora_surgery(base_model, target_modules=["c_attn"], r=4, lora_alpha=8)
print("\nSurgery complete! Let's verify trainable parameters...")

# 3. Calculate and verify parameter counts
total_params = 0
trainable_params = 0

for name, param in base_model.named_parameters():
    total_params += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters (LoRA weights only): {trainable_params:,}")
print(f"Percentage of model being trained: {(trainable_params / total_params) * 100:.4f}%")

Loading gpt2...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Surgery complete! Let's verify trainable parameters...
Total Parameters: 124,587,264
Trainable Parameters (LoRA weights only): 103,353,600
Percentage of model being trained: 82.9568%


In [12]:
from transformers import AutoModelForCausalLM

# 1. Load the model
model_name = "gpt2"
print(f"Loading {model_name}...")
base_model = AutoModelForCausalLM.from_pretrained(model_name).cuda()

# === THE FIX: Freeze EVERY parameter in the base model upfront ===
for param in base_model.parameters():
    param.requires_grad = False

# 2. Run the surgery exactly like before
apply_lora_surgery(base_model, target_modules=["c_attn"], r=4, lora_alpha=8)
print("\nSurgery complete! Let's verify trainable parameters...")

# 3. Recalculate parameter counts
total_params = 0
trainable_params = 0

for name, param in base_model.named_parameters():
    total_params += param.numel()
    if param.requires_grad:
        trainable_params += param.numel()

print(f"Total Parameters: {total_params:,}")
print(f"Trainable Parameters (LoRA weights only): {trainable_params:,}")
print(f"Percentage of model being trained: {(trainable_params / total_params) * 100:.4f}%")

Loading gpt2...


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]


Surgery complete! Let's verify trainable parameters...
Total Parameters: 124,587,264
Trainable Parameters (LoRA weights only): 175,104
Percentage of model being trained: 0.1405%


In [14]:
import torch.optim as optim

# 1. Initialize a real tokenizer for GPT-2 to get actual text token inputs
tokenizer = AutoTokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

# 2. Create actual text inputs and generate a batch
text = ["Fine-tuning transformers from scratch feels incredible.", "LoRA math works perfectly."]
inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True).to("cuda")

# 3. Setup a standard optimizer tracking ONLY the trainable parameters
optimizer = optim.AdamW(filter(lambda p: p.requires_grad, base_model.parameters()), lr=1e-4)

# 4. Forward Pass
outputs = base_model(input_ids=inputs["input_ids"], attention_mask=inputs["attention_mask"], labels=inputs["input_ids"])
loss = outputs.loss

# 5. Backward Pass
optimizer.zero_grad()
loss.backward()

# 6. EXPLICIT PROOF: Verify gradients exist ONLY on LoRA weights
print(f"Current Loss: {loss.item():.4f}\n")
print("Checking gradients across layers...")

lora_grad_found = False
base_grad_found = False

for name, param in base_model.named_parameters():
    if "lora_" in name:
        if param.grad is not None:
            lora_grad_found = True
    elif "linear.weight" in name or "wte" in name: # original weights
        if param.grad is not None:
            base_grad_found = True

print(f"Did our Custom LoRA weights receive gradients? -> {lora_grad_found}")
print(f"Did the frozen pre-trained weights receive gradients? -> {base_grad_found}")

Current Loss: 6.3205

Checking gradients across layers...
Did our Custom LoRA weights receive gradients? -> True
Did the frozen pre-trained weights receive gradients? -> False
